# 03 — Model development and freeze

This notebook reads only the `search_train`, `hp_validation`, and
`model_selection` feature partitions. It never opens the calibration or final
evaluation partitions. Hyperparameters are screened on a deterministic 10%
sample, surviving configurations are refit on all 2017–2020 rows, and
model/feature candidates are compared on deterministic 10% samples of the
2017–2021 training pool and 2022 selection partition.

The winning recipe is frozen for Notebook 04, which refits it on 100% of valid
2017–2022 rows. Calibration uses a deterministic 10% sample of 2023. No
training-fraction scaling study is run.

In [ ]:
import hashlib
import json
import os
import subprocess
import sys
import tempfile
import time
from pathlib import Path

from IPython.display import display
from pyspark import StorageLevel
from pyspark.sql import SparkSession, functions as F

BUCKET = os.environ.get("BUCKET", "sg-rainfall-dsa5208-261011-data")
RUN_ID = os.environ.get("RUN_ID", "20261016-v2-fixed8-04")
ROOT = f"gs://{BUCKET}/runs/{RUN_ID}"
RESULTS_ROOT = f"{ROOT}/results"
MODEL_ROOT = f"{ROOT}/models"


def gcloud(args):
    return subprocess.run(
        ["gcloud", *args], check=True, capture_output=True, text=True
    )


def read_gcs_text(uri):
    return gcloud(["storage", "cat", uri]).stdout


config_raw = read_gcs_text(f"{ROOT}/config/experiment.json")
config = json.loads(config_raw)
source_hashes_raw = read_gcs_text(f"{ROOT}/config/source_hashes.json")
source_hashes = json.loads(source_hashes_raw)
input_manifest_raw = read_gcs_text(f"{ROOT}/config/input_manifest.json")
config_sha256 = hashlib.sha256(config_raw.encode("utf-8")).hexdigest()
source_hashes_sha256 = hashlib.sha256(source_hashes_raw.encode("utf-8")).hexdigest()
input_manifest_sha256 = hashlib.sha256(input_manifest_raw.encode("utf-8")).hexdigest()


def notebook_source_sha256(uri):
    local = module_path.parent / Path(uri).name
    gcloud(["storage", "cp", uri, str(local)])
    notebook = json.loads(local.read_text(encoding="utf-8"))
    source_only = {
        "cells": [
            {
                "cell_type": cell["cell_type"],
                "source": (
                    "".join(cell.get("source", []))
                    if isinstance(cell.get("source", ""), list)
                    else cell.get("source", "")
                ),
            }
            for cell in notebook["cells"]
        ]
    }
    canonical = json.dumps(
        source_only, ensure_ascii=False, sort_keys=True, separators=(",", ":")
    ).encode("utf-8")
    return hashlib.sha256(canonical).hexdigest()


assert config["protocol_version"] == 2
assert config["split_names"] == [
    "search_train", "hp_validation", "model_selection", "calibration", "test"
]
assert config["final_training_fraction"] == 1.0
assert config["calibration_method"] == "isotonic"
assert config["monotonic_projection"] == "pava"
assert config["threshold_interpolation"] == "linear"

module_path = Path(tempfile.gettempdir()) / "dsa5208-rainfall-v2" / "rainfall_modeling.py"
module_path.parent.mkdir(parents=True, exist_ok=True)
gcloud(["storage", "cp", f"{ROOT}/source/rainfall_modeling.py", str(module_path)])
module_sha256 = hashlib.sha256(module_path.read_bytes()).hexdigest()
assert module_sha256 == source_hashes["scripts/rainfall_modeling.py"]
for relative_path in (
    "notebooks/02_build_features.ipynb",
    "notebooks/03_model_development.ipynb",
):
    actual_hash = notebook_source_sha256(f"{ROOT}/source/{relative_path}")
    assert actual_hash == source_hashes[relative_path], relative_path

sys.path.insert(0, str(module_path.parent))
from rainfall_modeling import (
    build_pipeline,
    choose_candidate,
    climatology_scores,
    deterministic_sample,
    expand_thresholds,
    feature_columns,
    fit_climatology_tables,
    mark_unseen,
    score_model,
    selection_objective,
    threshold_metrics,
    threshold_units,
)

spark = SparkSession.builder.appName("dsa5208-rainfall-v2-development").getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", str(config["shuffle_partitions"]))
spark.conf.set("spark.sql.session.timeZone", config["timezone"])
spark.conf.set("spark.sql.ansi.enabled", "false")
actual_spark_conf = {
    name: spark.conf.get(name)
    for name in (
        "spark.sql.shuffle.partitions",
        "spark.sql.session.timeZone",
        "spark.sql.ansi.enabled",
    )
}
assert actual_spark_conf["spark.sql.shuffle.partitions"] == str(config["shuffle_partitions"])
assert actual_spark_conf["spark.sql.session.timeZone"] == config["timezone"]
assert actual_spark_conf["spark.sql.ansi.enabled"] == "false"
display({
    "run_id": RUN_ID,
    "config_sha256": config_sha256,
    "input_manifest_sha256": input_manifest_sha256,
    "modeling_source_sha256": module_sha256,
    "spark_conf": actual_spark_conf,
})

In [ ]:
# Load only the three development partitions by their exact GCS paths.
def read_development_partition(name):
    if name not in {"search_train", "hp_validation", "model_selection"}:
        raise ValueError("Development notebook may read only development partitions.")
    return (
        spark.read.parquet(f"{ROOT}/base_features/split={name}")
        .where(F.col("label_valid"))
        .persist(StorageLevel.DISK_ONLY)
    )


search_train = read_development_partition("search_train")
hp_validation = read_development_partition("hp_validation")
model_selection = read_development_partition("model_selection")
development_rows = {
    "search_train_2017_2020": search_train.count(),
    "hp_validation_2021": hp_validation.count(),
    "model_selection_2022": model_selection.count(),
}
if any(value == 0 for value in development_rows.values()):
    raise ValueError(f"A required development partition is empty: {development_rows}")

thresholds = [float(value) for value in config["thresholds_mm"]]
support_aggregates = [
    F.sum(
        (F.col("future_units") > threshold_units(
            value, config["rainfall_integer_scale"]
        )).cast("long")
    ).alias(f"positive_{index}")
    for index, value in enumerate(thresholds)
]
support_row = search_train.agg(F.count("*").alias("n"), *support_aggregates).first().asDict()
primary_thresholds = [
    value for index, value in enumerate(thresholds)
    if int(support_row[f"positive_{index}"] or 0) >= config["primary_threshold_min_positives"]
    and int(support_row[f"positive_{index}"] or 0) < int(support_row["n"])
]
if not primary_thresholds:
    raise ValueError("No threshold has both classes and the required positive support.")
display({
    "valid_base_rows": development_rows,
    "primary_thresholds_mm": primary_thresholds,
    "screening_fraction": config["tuning_sample_fraction"],
    "feature_counts": {
        "A": len(feature_columns(config, "A")),
        "B": len(feature_columns(config, "B")),
    },
})

In [ ]:
# Screen hyperparameters on the deterministic 10% sample, then refit survivors
# on all 2017–2020 search-training rows and compare them on 2021.
def parameter_options(algorithm):
    if algorithm == "lr":
        return [{"regParam": float(value)} for value in config["lr_reg_params"]]
    if algorithm == "rf":
        return config["rf_configs"]
    if algorithm == "gbt":
        return config["gbt_configs"]
    raise ValueError(f"Unknown algorithm: {algorithm}")


screen_base = deterministic_sample(
    search_train, config["tuning_sample_fraction"], config["sampling_seed"]
).persist(StorageLevel.DISK_ONLY)
screen_expanded = expand_thresholds(screen_base, config).persist(StorageLevel.DISK_ONLY)
screen_stations = screen_base.select("station_id").distinct().persist(StorageLevel.DISK_ONLY)
screen_validation = mark_unseen(
    expand_thresholds(hp_validation, config), screen_stations
).persist(StorageLevel.DISK_ONLY)

screen_records = []
for algorithm in ("lr", "rf", "gbt"):
    variant = "A" if algorithm == "lr" else "B"
    for index, parameters in enumerate(parameter_options(algorithm)):
        started = time.perf_counter()
        fitted = build_pipeline(algorithm, parameters, variant, config).fit(screen_expanded)
        fit_seconds = time.perf_counter() - started
        predictions = score_model(fitted, screen_validation).persist(StorageLevel.DISK_ONLY)
        screen_records.append({
            "candidate_id": f"{algorithm}_screen_{index}",
            "algorithm": algorithm,
            "variant": variant,
            "parameters": parameters,
            "fit_seconds": fit_seconds,
            "metrics": threshold_metrics(
                predictions, config["thresholds_mm"], config
            ),
        })
        predictions.unpersist()
        print(f"Screened {algorithm} configuration {index} in {fit_seconds:.1f}s")

# Keep at most two configurations per family that are within the predeclared
# PR-AUC tolerance of that family's best screening score.
survivors = []
for algorithm in ("lr", "rf", "gbt"):
    family = [
        record for record in screen_records if record["algorithm"] == algorithm
    ]
    ranked = [
        (record, *selection_objective(record["metrics"], primary_thresholds))
        for record in family
    ]
    eligible = [item for item in ranked if item[1] is not None]
    if not eligible:
        raise ValueError(f"No screened {algorithm} configuration has supported metrics.")
    best_auc = max(item[1] for item in eligible)
    close = [
        item for item in eligible
        if best_auc - item[1] <= config["selection_pr_auc_tolerance"]
    ]
    close.sort(key=lambda item: (item[2], item[0]["candidate_id"]))
    survivors.extend(item[0] for item in close[:2])

full_search_expanded = expand_thresholds(search_train, config).persist(StorageLevel.DISK_ONLY)
full_search_stations = search_train.select("station_id").distinct().persist(StorageLevel.DISK_ONLY)
full_validation = mark_unseen(
    expand_thresholds(hp_validation, config), full_search_stations
).persist(StorageLevel.DISK_ONLY)

full_hp_records = []
for record in survivors:
    started = time.perf_counter()
    fitted = build_pipeline(
        record["algorithm"], record["parameters"], record["variant"], config
    ).fit(full_search_expanded)
    fit_seconds = time.perf_counter() - started
    predictions = score_model(fitted, full_validation).persist(StorageLevel.DISK_ONLY)
    full_hp_records.append({
        "candidate_id": record["candidate_id"].replace("_screen_", "_full_"),
        "screen_candidate_id": record["candidate_id"],
        "algorithm": record["algorithm"],
        "variant": record["variant"],
        "parameters": record["parameters"],
        "fit_seconds": fit_seconds,
        "metrics": threshold_metrics(
            predictions, config["thresholds_mm"], config
        ),
    })
    predictions.unpersist()
    print(f"Full 2017–2020 refit completed for {record['candidate_id']}")

best_parameters = {}
for algorithm in ("lr", "rf", "gbt"):
    family = [row for row in full_hp_records if row["algorithm"] == algorithm]
    best_parameters[algorithm] = choose_candidate(
        family, primary_thresholds, config
    )["parameters"]

display({
    "screened_configurations": len(screen_records),
    "full_refit_survivors": len(full_hp_records),
    "selected_hyperparameters": best_parameters,
})

In [ ]:
# Compare model families/features on deterministic 10% samples, then freeze the winner.
model_selection_fraction = float(config["model_selection_sample_fraction"])
training_pool = search_train.unionByName(hp_validation).persist(StorageLevel.DISK_ONLY)
selection_training_sample = deterministic_sample(
    training_pool, model_selection_fraction, config["sampling_seed"]
).persist(StorageLevel.DISK_ONLY)
selection_training_base_rows = selection_training_sample.count()
if selection_training_base_rows == 0:
    raise ValueError("The model-selection training sample is empty.")

selection_evaluation_sample = deterministic_sample(
    model_selection, model_selection_fraction, config["sampling_seed"]
).persist(StorageLevel.DISK_ONLY)
selection_evaluation_base_rows = selection_evaluation_sample.count()
if selection_evaluation_base_rows == 0:
    raise ValueError("The model-selection evaluation sample is empty.")

model_selection_stations = selection_training_sample.select("station_id").distinct().persist(
    StorageLevel.DISK_ONLY
)
training_pool_expanded = expand_thresholds(
    selection_training_sample, config
).persist(StorageLevel.DISK_ONLY)
selection_queries = mark_unseen(
    expand_thresholds(selection_evaluation_sample, config), model_selection_stations
).persist(StorageLevel.DISK_ONLY)

baseline_tables = fit_climatology_tables(selection_training_sample)
baseline_selection = climatology_scores(
    selection_queries, baseline_tables, config["climatology_smoothing"]
).persist(StorageLevel.DISK_ONLY)
selection_records = [{
    "candidate_id": "climatology",
    "algorithm": "climatology",
    "variant": "baseline",
    "parameters": {},
    "fit_seconds": 0.0,
    "metrics": threshold_metrics(
        baseline_selection, config["thresholds_mm"], config
    ),
}]
development_models = {}
development_fit_seconds = {}

for algorithm, variants in (
    ("lr", ("A",)),
    ("rf", ("B",)),
    ("gbt", ("A", "B")),
):
    for variant in variants:
        candidate_id = f"{algorithm}_{variant}"
        started = time.perf_counter()
        fitted = build_pipeline(
            algorithm, best_parameters[algorithm], variant, config
        ).fit(training_pool_expanded)
        development_fit_seconds[candidate_id] = time.perf_counter() - started
        development_models[candidate_id] = fitted
        predictions = score_model(fitted, selection_queries).persist(
            StorageLevel.DISK_ONLY
        )
        selection_records.append({
            "candidate_id": candidate_id,
            "algorithm": algorithm,
            "variant": variant,
            "parameters": best_parameters[algorithm],
            "fit_seconds": development_fit_seconds[candidate_id],
            "metrics": threshold_metrics(
                predictions, config["thresholds_mm"], config
            ),
        })
        predictions.unpersist()
        print(
            f"Fitted {candidate_id} on {selection_training_base_rows:,} sampled "
            f"2017–2021 rows in {development_fit_seconds[candidate_id]:.1f}s"
        )

selected = choose_candidate(selection_records, primary_thresholds, config)
selected_name = selected["candidate_id"]
selected_pr_auc, selected_brier = selection_objective(
    selected["metrics"], primary_thresholds
)

development_summary = {
    "protocol_version": 2,
    "valid_base_rows": development_rows,
    "primary_thresholds_mm": primary_thresholds,
    "hyperparameter_screening": {
        "sample_fraction": config["tuning_sample_fraction"],
        "seed": config["sampling_seed"],
        "screen_records": screen_records,
        "survivors_refit_on_full_search_train": full_hp_records,
        "selected_parameters": best_parameters,
        "validation_split": "hp_validation_2021",
    },
    "model_selection": {
        "training_partitions": ["search_train_2017_2020", "hp_validation_2021"],
        "training_sample_fraction": model_selection_fraction,
        "training_base_rows": selection_training_base_rows,
        "evaluation_split": "model_selection_2022",
        "evaluation_sample_fraction": model_selection_fraction,
        "evaluation_base_rows": selection_evaluation_base_rows,
        "candidates": selection_records,
        "selected_model": selected_name,
        "selected_mean_pr_auc": selected_pr_auc,
        "selected_mean_brier": selected_brier,
    },
}
display({
    "selected_model": selected_name,
    "selected_mean_pr_auc": selected_pr_auc,
    "selected_mean_brier": selected_brier,
    "model_selection_training_sample_fraction": model_selection_fraction,
    "model_selection_training_base_rows": selection_training_base_rows,
    "model_selection_evaluation_sample_fraction": model_selection_fraction,
    "model_selection_evaluation_base_rows": selection_evaluation_base_rows,
})

In [ ]:
# Save the development evidence and create the immutable contract for Notebook 04.
from datetime import datetime, timezone

def upload_json(uri, payload):
    local_path = Path(tempfile.gettempdir()) / f"dsa5208-{RUN_ID}-{Path(uri).name}"
    local_path.write_text(
        json.dumps(payload, indent=2, sort_keys=True, allow_nan=False) + "\n",
        encoding="utf-8",
    )
    try:
        gcloud(["storage", "cp", str(local_path), uri])
    finally:
        local_path.unlink(missing_ok=True)


spec_uri = f"{ROOT}/artifacts/frozen_spec.json"
existing_spec = subprocess.run(
    ["gcloud", "storage", "objects", "describe", spec_uri, "--format=json"],
    capture_output=True,
    text=True,
)
if existing_spec.returncode == 0:
    raise FileExistsError(f"Frozen specification already exists: {spec_uri}")
if "not found" not in existing_spec.stderr.lower() and "no objects" not in existing_spec.stderr.lower():
    raise RuntimeError(
        "Could not confirm that the frozen specification path is absent; "
        "inspect the GCS error before continuing."
    )

frozen_spec = {
    "protocol_version": 2,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "selected_model": selected_name,
    "algorithm": selected["algorithm"],
    "variant": selected["variant"],
    "parameters": selected["parameters"],
    "selection_split": "model_selection_2022",
    "selection_sample_fraction": model_selection_fraction,
    "selection_training_base_rows": selection_training_base_rows,
    "selection_evaluation_base_rows": selection_evaluation_base_rows,
    "selection_mean_pr_auc": selected_pr_auc,
    "selection_mean_brier": selected_brier,
    "hyperparameter_validation_split": "hp_validation_2021",
    "hyperparameters": best_parameters,
    "final_training_partitions": [
        "search_train_2017_2020", "hp_validation_2021", "model_selection_2022"
    ],
    "final_training_fraction": 1.0,
    "thresholds_mm": thresholds,
    "primary_thresholds_mm": primary_thresholds,
    "threshold_domain_mm": config["threshold_domain_mm"],
    "calibration": {
        "method": "isotonic",
        "partition": "calibration_2023",
        "sample_fraction": config["calibration_sample_fraction"],
        "sampling_seed": config["sampling_seed"],
        "always_applied": True,
    },
    "monotonic_projection": {
        "method": "equal_weight_pava_nonincreasing",
        "anchors_mm": thresholds,
        "applied_after_calibration": True,
    },
    "threshold_interpolation": {
        "method": "linear",
        "domain_inclusive_mm": config["threshold_domain_mm"],
        "outside_domain": "reject",
    },
    "reference_probability_cutoff": 0.5,
    "experiment_config_sha256": config_sha256,
    "input_manifest_sha256": input_manifest_sha256,
    "source_hashes_manifest_sha256": source_hashes_sha256,
    "feature_source_sha256": source_hashes["notebooks/02_build_features.ipynb"],
    "development_notebook_sha256": source_hashes[
        "notebooks/03_model_development.ipynb"
    ],
    "final_notebook_sha256": source_hashes["notebooks/04_final_fit_evaluate.ipynb"],
    "modeling_source_sha256": module_sha256,
    "historical_v1_2024_metrics_previously_observed": True,
    "v2_2024_label": "held-out evaluation period",
}
upload_json(f"{ROOT}/results/development_summary.json", development_summary)
upload_json(spec_uri, frozen_spec)

print({
    "development_summary": f"{ROOT}/results/development_summary.json",
    "frozen_spec": spec_uri,
    "frozen_spec_sha256": hashlib.sha256(
        json.dumps(frozen_spec, sort_keys=True, allow_nan=False).encode("utf-8")
    ).hexdigest(),
    "final_training_fraction": frozen_spec["final_training_fraction"],
    "selected_model": selected_name,
    "test_partition_opened": False,
})

for frame in (
    search_train, hp_validation, model_selection, screen_base, screen_expanded,
    screen_stations, screen_validation, full_search_expanded, full_search_stations,
    full_validation, training_pool, model_selection_stations, training_pool_expanded,
    selection_training_sample, selection_evaluation_sample, selection_queries,
    baseline_selection,
):
    frame.unpersist()
spark.stop()